## Fine Tune ResNet-18 weights pretrained on Sentinel-2 RGB imagery

In [1]:
%pip install torch torchvision

  Using cached torchvision-0.29.2-py3-none-win_amd64.whl.metadata (5.7 kB)
  Using cached filelock-4.2.12-py3-none-any.whl.metadata (2.3 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
    --------------------------------------- 1.8/126.7 MB 11.2 MB/s eta 0:00:12
   - -------------------------------------- 4.2/126.7 MB 10.5 MB/s eta 0:00:12
   -- ------------------------------------- 6.8/126.7 MB 11.3 MB/s eta 0:00:11
   --- ------------------------------------ 10.0/126.7 MB 12.4 MB/s eta 0:00:10
   ---- ----------------------------------- 13.1/126.7 MB 13.0 MB/s eta 0:00:09
   ----


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install torchgeo timm

  Using cached torchgeo-0.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached timm-1.0.31-py3-none-any.whl.metadata (42 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached geopandas-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached jsonargparse-4.53.0-py3-none-any.whl.metadata (13 kB)
  Using cached kornia-0.9.2-py3-none-any.whl.metadata (25 kB)
  Using cached lightly-1.6.1-py3-none-any.whl.metadata (45 kB)
  Using cached lightning-2.6.6-py3-none-any.whl.metadata (45 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-win_amd64.whl.metadata (6.0 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached segmentation_models_pytorch-0.6.0-py3-none-any.whl.metadata (16 kB)
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadat


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 4.2 MB/s eta 0:00:02
   ------ --------------------------------- 1.3/8.3 MB 3.7 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/8.3 MB 3.9 MB/s eta 0:00:02
   ------------- -------------------------- 2.9/8.3 MB 3.6 MB/s eta 0:00:02
   ----------------- ---------------------- 3.7/8.3 MB 3.6 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.3 MB 3.7 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.3 MB 3.6 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.3 MB 3.6 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 3.5 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.3 MB 3.4 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.3 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 3.3 MB/s  0:00:02
   -------------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch
import torch.nn as nn
from torchgeo.models import ResNet18_Weights, resnet18
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

Matplotlib is building the font cache; this may take a moment.
c:\Users\Emily\OneDrive\Pictures\Saved Pictures\Documents\GitHub\dst-sprint-2\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import time
import timm

Needed to download an older version of python (3.12) to be able to user torchgeo

Parts to explore:
- Replace output head
- Freezing early layers
- Adding layers
- Optimal learning rate
- Regularisation (Early stopping/Dropout)

Reccommeneded to use lower learning rate then during training to avoid making large changes to the model

Resnet-18 is a Convolutional Neural Network (CNN) used mainly for image classification. It was introduced in the 2015 paper Deep Residual Learning for Image Recognition. Its main distinct feature is the use of residual blocks with skip connections.

### SSL4EO-S12 Sentinel-2 RGB MoCo (ResNet-18) Architecture

| Layer | Operation | Configuration | Output Shape |
|---|---|---|---|
| Input | RGB Image | 3 channels, 224 × 224 | 3 × 224 × 224 |
| Conv1 | Convolution + BatchNorm + ReLU | 7 × 7, 64 filters, stride 2 | 64 × 112 × 112 |
| MaxPool | Max Pooling | 3 × 3, stride 2 | 64 × 56 × 56 |
| Layer 1 | 2 × BasicBlock | 64 channels | 64 × 56 × 56 |
| Layer 2 | 2 × BasicBlock | 128 channels, stride 2 | 128 × 28 × 28 |
| Layer 3 | 2 × BasicBlock | 256 channels, stride 2 | 256 × 14 × 14 |
| Layer 4 | 2 × BasicBlock | 512 channels, stride 2 | 512 × 7 × 7 |
| Global Average Pooling | Adaptive Average Pooling | 7 × 7 → 1 × 1 | 512 |
| Fully Connected (FC) | Linear Layer | 512 → 1000 | 1000 |

### Model Details

| Property | Description |
|---|---|
| Architecture | ResNet-18 |
| Pretraining Dataset | SSL4EO-S12 |
| Pretraining Method | MoCo |
| Satellite | Sentinel-2 |
| Input Bands | RGB (B04, B03, B02) |
| Input Channels | 3 |
| Number of Residual Blocks | 8 |
| Feature Dimension | 512 |
| Final Layer | Linear(512, 1000) |
| Final Layer Pretrained | No, randomly initialised |
| Output Dimension | 1000 |
| Output Classes | No predefined land-cover classes |

**Note:** The ResNet-18 backbone is pretrained using MoCo. The original MoCo projection head is used during self-supervised pretraining and is not part of the standard TorchGeo ResNet-18 model returned for classification.

## Data

In [12]:
# Step 1: Load training images temporarily without normalisation
initial_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

initial_train_dataset = datasets.ImageFolder(
    root="../../data/train",
    transform=initial_transform
)

# Step 2: Calculate RGB mean and standard deviation
stats_loader = DataLoader(
    initial_train_dataset,
    batch_size=32,
    shuffle=False
)

channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
total_pixels = 0

for images, labels in stats_loader:
    channel_sum += images.sum(dim=[0, 2, 3])
    channel_squared_sum += (images ** 2).sum(dim=[0, 2, 3])
    total_pixels += images.shape[0] * images.shape[2] * images.shape[3]

mean = channel_sum / total_pixels
std = torch.sqrt(channel_squared_sum / total_pixels - mean ** 2)

print("RGB mean:", mean.tolist())
print("RGB standard deviation:", std.tolist())

# Step 3: Create preprocessing transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])

# Step 4: Load the final training and test datasets
train_dataset = datasets.ImageFolder(
    root="../../data/train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="../../data/test",
    transform=transform
)

# Step 5: Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# Step 6: Check the datasets
print("\nClasses:", train_dataset.classes)
print("Number of classes:", len(train_dataset.classes))
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))

# Check a batch of images
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)


RGB mean: [0.3446079194545746, 0.38057249784469604, 0.4081569314002991]
RGB standard deviation: [0.201094850897789, 0.1351122111082077, 0.11359146237373352]

Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Number of classes: 10
Training images: 21600
Test images: 5400
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


# Base model: only tuned final layer

In [18]:
import numpy as np
from torch.utils.data import Subset, DataLoader


# Set random seed for reproducibility
np.random.seed(42)

# Select 20% of images from each class
subset_indices = []

for class_index, class_name in enumerate(train_dataset.classes):

    # Find all image indices belonging to this class
    class_indices = np.where(
        np.array(train_dataset.targets) == class_index
    )[0]

    # Randomly select 20% of that class
    selected_count = int(len(class_indices) * 0.20)

    selected_indices = np.random.choice(
        class_indices,
        size=selected_count,
        replace=False
    )

    subset_indices.extend(selected_indices)

# Create the representative training subset
train_subset = Subset(train_dataset, subset_indices)

# Create DataLoaders
train_subset_loader = DataLoader(
    train_subset,
    batch_size=32,
    shuffle=True
)



print("Original training images:", len(train_dataset))
print("20% training subset:", len(train_subset))
print("Full test dataset:", len(test_dataset))


Original training images: 21600
20% training subset: 4320
Full test dataset: 5400


Load pre-trained weights for tuning of final layer only

In [32]:
import torch
import torch.nn as nn

from torchgeo.models import ResNet18_Weights, resnet18

# Load pretrained Sentinel-2 RGB MoCo weights
weights = ResNet18_Weights.SENTINEL2_RGB_MOCO

model = resnet18(weights=weights)

# Freeze pretrained backbone
for param in model.parameters():
    param.requires_grad = False

# Replace final classification layer
model.fc = nn.Linear(model.fc.in_features, 10)

# Configure loss and optimiser
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print("Model configured for baseline training.")

Model configured for baseline training.


In [34]:
# Save the original pretrained weights
pretrained_weights = copy.deepcopy(model.state_dict())

In [21]:
import time
import torch

# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

epochs = 3

history = {
    "train_loss": [],
    "train_accuracy": []
}

# Start overall training timer
start_time = time.time()

for epoch in range(epochs):

    epoch_start = time.time()

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_subset_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

    # Calculate training metrics
    epoch_loss = running_loss / len(train_subset_loader)
    epoch_accuracy = 100 * correct / total

    history["train_loss"].append(epoch_loss)
    history["train_accuracy"].append(epoch_accuracy)

    # Calculate epoch time
    epoch_time = time.time() - epoch_start

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Loss: {epoch_loss:.4f} | "
        f"Training accuracy: {epoch_accuracy:.2f}% | "
        f"Time: {epoch_time / 60:.2f} minutes"
    )

    print("-" * 60)

# Calculate total training time
total_time = time.time() - start_time

print("\nTraining completed!")
print(f"Total training time: {total_time / 60:.2f} minutes")



Epoch [1/3] | Loss: 0.8530 | Training accuracy: 88.08% | Time: 8.01 minutes
------------------------------------------------------------
Epoch [2/3] | Loss: 0.7153 | Training accuracy: 89.12% | Time: 8.09 minutes
------------------------------------------------------------
Epoch [3/3] | Loss: 0.6220 | Training accuracy: 90.16% | Time: 9.44 minutes
------------------------------------------------------------

Training completed!
Total training time: 25.55 minutes


In [22]:
torch.save(model.state_dict(), "eurosat_baseline.pth")

In [30]:
import sklearn
from sklearn.metrics import classification_report, confusion_matrix

model.eval()

all_predictions = []
all_labels = []

evaluation_start = time.time()

with torch.inference_mode():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

evaluation_time = time.time() - evaluation_start

# Calculate final accuracy
final_accuracy = (
    sum(p == y for p, y in zip(all_predictions, all_labels))
    / len(all_labels)
) * 100

print(f"Final test accuracy: {final_accuracy:.2f}%")
print(f"Evaluation time: {evaluation_time:.2f} seconds")

# Classification report
print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=train_dataset.classes
    )
)

# Confusion matrix
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)


Final test accuracy: 90.56%
Evaluation time: 719.57 seconds

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.86      0.97      0.91       600
              Forest       0.91      0.99      0.95       600
HerbaceousVegetation       0.87      0.93      0.90       600
             Highway       0.83      0.71      0.77       500
          Industrial       0.92      0.97      0.94       500
             Pasture       0.92      0.88      0.90       400
       PermanentCrop       0.92      0.81      0.86       500
         Residential       0.98      0.98      0.98       600
               River       0.86      0.78      0.82       500
             SeaLake       0.98      0.97      0.97       600

            accuracy                           0.91      5400
           macro avg       0.90      0.90      0.90      5400
        weighted avg       0.91      0.91      0.90      5400


Confusion Matrix:
[[581   0   1   1   0   3 

Taking extremely long to run, need to use smaller amount

Tuning lr

In [35]:

import copy

# Learning rates to compare
learning_rates = [0.001, 0.01, 0.0001]

# Store results
results = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for lr in learning_rates:

    print(f"\nTraining with learning rate: {lr}")

    # Create a fresh copy of the pretrained model for each experiment
    model_lr = copy.deepcopy(model)
    model_lr.load_state_dict(pretrained_weights)

    # Freeze the backbone
    for param in model_lr.parameters():
        param.requires_grad = False

    # Make sure the final classification layer is trainable
    for param in model_lr.fc.parameters():
        param.requires_grad = True

    # Create a new optimiser for this learning rate
    optimizer_lr = torch.optim.Adam(
        model_lr.fc.parameters(),
        lr=lr
    )

    # Train for 1 epoch
    model_lr.train()
    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for images, labels in train_subset_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_lr.zero_grad(set_to_none=True)

        outputs = model_lr(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_lr.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

    # Calculate results
    epoch_loss = running_loss / len(train_subset_loader)
    epoch_accuracy = 100 * correct / total
    training_time = time.time() - start_time

    results[lr] = {
        "loss": epoch_loss,
        "accuracy": epoch_accuracy,
        "time": training_time
    }

    print(f"Training loss: {epoch_loss:.4f}")
    print(f"Training accuracy: {epoch_accuracy:.2f}%")
    print(f"Training time: {training_time / 60:.2f} minutes")

print("\nLearning rate comparison completed!")


Training with learning rate: 0.001
Training loss: 1.9412
Training accuracy: 70.67%
Training time: 8.90 minutes

Training with learning rate: 0.01
Training loss: 0.8345
Training accuracy: 83.26%
Training time: 9.83 minutes

Training with learning rate: 0.0001
Training loss: 2.2541
Training accuracy: 42.48%
Training time: 10.23 minutes

Learning rate comparison completed!


In [36]:
import pandas as pd

results_df = pd.DataFrame(results).T
results_df.index.name = "Learning Rate"

print(results_df)

                   loss   accuracy        time
Learning Rate                                 
0.0010         1.941168  70.671296  533.931698
0.0100         0.834484  83.263889  589.559521
0.0001         2.254132  42.476852  614.038513


lr 0.01 according to research original lr is 0.03

Now trying also fine tune previous layer

Using lower lr for layer 4

In [44]:
# Load original pretrained ResNet-18

model_2 = copy.deepcopy(model)
model_2.load_state_dict(pretrained_weights)

<All keys matched successfully>

In [45]:
# Freeze all model parameters
for param in model_2.parameters():
    param.requires_grad = False

# Unfreeze final ResNet block
for param in model_2.layer4.parameters():
    param.requires_grad = True

# Unfreeze classification layer
for param in model_2.fc.parameters():
    param.requires_grad = True

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_2 = model_2.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam([
    {"params": model_2.layer4.parameters(), "lr": 0.0001},
    {"params": model_2.fc.parameters(), "lr": 0.01}
])

In [47]:

model_2.train()

running_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for images, labels in train_subset_loader:

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model_2(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

    predictions = outputs.argmax(dim=1)
    total += labels.size(0)
    correct += (predictions == labels).sum().item()

epoch_loss = running_loss / len(train_subset_loader)
epoch_accuracy = 100 * correct / total

print(f"Training loss: {epoch_loss:.4f}")
print(f"Training accuracy: {epoch_accuracy:.2f}%")
print(f"Training time: {(time.time() - start_time) / 60:.2f} minutes")

Training loss: 0.4370
Training accuracy: 87.71%
Training time: 13.30 minutes


We have to imagine this will x5xn when applied to n epochs on all test data

Change layer 3

In [49]:
# Load original pretrained ResNet-18

model_3 = copy.deepcopy(model)
model_3.load_state_dict(pretrained_weights)

<All keys matched successfully>

In [50]:
# Freeze all model parameters
for param in model_3.parameters():
    param.requires_grad = False

# Unfreeze final ResNet block
for param in model_3.layer3.parameters():
    param.requires_grad = True

# Unfreeze final ResNet block
for param in model_3.layer4.parameters():
    param.requires_grad = True

# Unfreeze classification layer
for param in model_3.fc.parameters():
    param.requires_grad = True

In [51]:
model_3.train()

running_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for images, labels in train_subset_loader:

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model_3(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

    predictions = outputs.argmax(dim=1)
    total += labels.size(0)
    correct += (predictions == labels).sum().item()

epoch_loss = running_loss / len(train_subset_loader)
epoch_accuracy = 100 * correct / total

print(f"Training loss: {epoch_loss:.4f}")
print(f"Training accuracy: {epoch_accuracy:.2f}%")
print(f"Training time: {(time.time() - start_time) / 60:.2f} minutes")

Training loss: 2.2945
Training accuracy: 17.57%
Training time: 16.78 minutes


Imagine + mins x15 for 3 epochs on full training data for only pct increase in accuracy not worth it when scaled.

lora

In [52]:
%pip install loralib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:

import loralib as lora


from torchgeo.models import ResNet18_Weights, resnet18

# Load original pretrained model
model_lora = resnet18(
    weights=ResNet18_Weights.SENTINEL2_RGB_MOCO
)

# Freeze original pretrained parameters
for param in model_lora.parameters():
    param.requires_grad = False

# Replace the final classification layer
model_lora.fc = nn.Linear(model_lora.fc.in_features, 10)

In [54]:
import torch
import torch.nn as nn
import math

class LoRAConv2d(nn.Module):
    def __init__(self, original_conv, rank=4, alpha=8):
        super().__init__()

        if original_conv.groups != 1:
            raise ValueError("This implementation supports groups=1 only.")

        # Keep the original pretrained convolution
        self.base_conv = original_conv

        for param in self.base_conv.parameters():
            param.requires_grad = False

        # LoRA A: same kernel size and stride as original convolution
        self.lora_A = nn.Conv2d(
            in_channels=original_conv.in_channels,
            out_channels=rank,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            dilation=original_conv.dilation,
            bias=False,
            padding_mode=original_conv.padding_mode
        )

        # LoRA B: 1x1 convolution
        self.lora_B = nn.Conv2d(
            in_channels=rank,
            out_channels=original_conv.out_channels,
            kernel_size=1,
            bias=False
        )

        # Match original device and data type
        self.lora_A.to(
            device=original_conv.weight.device,
            dtype=original_conv.weight.dtype
        )
        self.lora_B.to(
            device=original_conv.weight.device,
            dtype=original_conv.weight.dtype
        )

        # Initialise LoRA weights
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

        self.scaling = alpha / rank

    def forward(self, x):
        original_output = self.base_conv(x)

        lora_output = self.lora_B(self.lora_A(x))

        return original_output + self.scaling * lora_output
import math

class LoRAConv2d(nn.Module):
    def __init__(self, original_conv, rank=4, alpha=8):
        super().__init__()

        if original_conv.groups != 1:
            raise ValueError("This implementation supports groups=1 only.")

        # Keep the original pretrained convolution
        self.base_conv = original_conv

        for param in self.base_conv.parameters():
            param.requires_grad = False

        # LoRA A: same kernel size and stride as original convolution
        self.lora_A = nn.Conv2d(
            in_channels=original_conv.in_channels,
            out_channels=rank,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            dilation=original_conv.dilation,
            bias=False,
            padding_mode=original_conv.padding_mode
        )

        # LoRA B: 1x1 convolution
        self.lora_B = nn.Conv2d(
            in_channels=rank,
            out_channels=original_conv.out_channels,
            kernel_size=1,
            bias=False
        )

        # Match original device and data type
        self.lora_A.to(
            device=original_conv.weight.device,
            dtype=original_conv.weight.dtype
        )
        self.lora_B.to(
            device=original_conv.weight.device,
            dtype=original_conv.weight.dtype
        )

        # Initialise LoRA weights
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

        self.scaling = alpha / rank

    def forward(self, x):
        original_output = self.base_conv(x)

        lora_output = self.lora_B(self.lora_A(x))

        return original_output + self.scaling * lora_output

In [55]:
def apply_lora(module, rank=4, alpha=8):

    for name, child in list(module.named_children()):

        if isinstance(child, nn.Conv2d):
            setattr(
                module,
                name,
                LoRAConv2d(child, rank=rank, alpha=alpha)
            )

        else:
            apply_lora(child, rank=rank, alpha=alpha)


# Apply LoRA only to layer4
apply_lora(model_lora.layer4, rank=4, alpha=8)

In [56]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_lora = model_lora.to(device)

criterion = nn.CrossEntropyLoss()

# Collect only LoRA and FC parameters
lora_parameters = [
    param
    for name, param in model_lora.named_parameters()
    if "lora_A" in name or "lora_B" in name
]

fc_parameters = list(model_lora.fc.parameters())

optimizer_lora = torch.optim.Adam([
    {"params": lora_parameters, "lr": 0.001},
    {"params": fc_parameters, "lr": 0.01}
])

print("Trainable LoRA parameters:", sum(
    p.numel() for p in lora_parameters
))

print("Trainable FC parameters:", sum(
    p.numel() for p in fc_parameters
))

Trainable LoRA parameters: 75776
Trainable FC parameters: 5130


In [57]:
import time

start_time = time.time()

model_lora.train()

# Keep BatchNorm running statistics fixed
for module in model_lora.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()

running_loss = 0.0
correct = 0
total = 0

for images, labels in train_subset_loader:

    images = images.to(device)
    labels = labels.to(device)

    optimizer_lora.zero_grad(set_to_none=True)

    outputs = model_lora(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer_lora.step()

    running_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    total += labels.size(0)
    correct += (predictions == labels).sum().item()

epoch_loss = running_loss / len(train_subset_loader)
epoch_accuracy = 100 * correct / total

training_time = time.time() - start_time

print("LoRA training completed!")
print(f"Training loss: {epoch_loss:.4f}")
print(f"Training accuracy: {epoch_accuracy:.2f}%")
print(f"Training time: {training_time / 60:.2f} minutes")

LoRA training completed!
Training loss: 0.9551
Training accuracy: 65.83%
Training time: 9.42 minutes


lora on 3 and 4

In [58]:

model_lora34 = resnet18(
    weights=ResNet18_Weights.SENTINEL2_RGB_MOCO
)

# Freeze original pretrained parameters
for param in model_lora34.parameters():
    param.requires_grad = False

# Replace classification layer
model_lora34.fc = nn.Linear(
    model_lora34.fc.in_features, 10
)

In [59]:
# Apply LoRA to layer3 and layer4
apply_lora(model_lora34.layer3, rank=4, alpha=8)
apply_lora(model_lora34.layer4, rank=4, alpha=8)

In [60]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_lora34 = model_lora34.to(device)

criterion = nn.CrossEntropyLoss()

# Collect LoRA parameters
lora_parameters = [
    param
    for name, param in model_lora34.named_parameters()
    if "lora_A" in name or "lora_B" in name
]

# Collect classification layer parameters
fc_parameters = list(model_lora34.fc.parameters())

optimizer_lora34 = torch.optim.Adam([
    {"params": lora_parameters, "lr": 0.001},
    {"params": fc_parameters, "lr": 0.01}
])

print("Trainable LoRA parameters:",
      sum(p.numel() for p in lora_parameters))

print("Trainable FC parameters:",
      sum(p.numel() for p in fc_parameters))

Trainable LoRA parameters: 113664
Trainable FC parameters: 5130


In [61]:
start_time = time.time()

model_lora34.train()

# Keep BatchNorm running statistics fixed
for module in model_lora34.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()

running_loss = 0.0
correct = 0
total = 0

for images, labels in train_subset_loader:

    images = images.to(device)
    labels = labels.to(device)

    optimizer_lora34.zero_grad(set_to_none=True)

    outputs = model_lora34(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer_lora34.step()

    running_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    total += labels.size(0)
    correct += (predictions == labels).sum().item()

epoch_loss = running_loss / len(train_subset_loader)
epoch_accuracy = 100 * correct / total

training_time = time.time() - start_time

print("Layer3 + Layer4 LoRA training completed!")
print(f"Training loss: {epoch_loss:.4f}")
print(f"Training accuracy: {epoch_accuracy:.2f}%")
print(f"Training time: {training_time / 60:.2f} minutes")

Layer3 + Layer4 LoRA training completed!
Training loss: 0.9087
Training accuracy: 68.15%
Training time: 10.88 minutes
